In [3]:
import os, requests, json

url = os.environ["OPENWEBUI_URI"].rstrip("/") + "/ollama/api/generate"  # → …/generate
headers = {
    "Authorization": f"Bearer {os.environ['OPENWEBUI_API_KEY']}",
    "Content-Type": "application/json"
}
payload = {
    "model": "llama3.2",
    "prompt": "What color is the sky at different times of the day? Respond using JSON",
    "format": "json",
    "stream": False
}

resp = requests.post(url, headers=headers, json=payload, timeout=60)
resp.raise_for_status()                       # raise if HTTP error
print(json.dumps(resp.json(), indent=2))


JSONDecodeError: Expecting value: line 1 column 1 (char 0)

In [4]:
import os

print(os.getenv("OPENWEBUI_URI"))
print(os.getenv("OPENWEBUI_API_KEY"))

https://ai.forge.unibw.de
sk-b6470cada6a74d29929ce01b4a268830


## In-place config vs Import from modules

both should work the same

In [6]:
from openai import AsyncOpenAI
from pydantic_ai import Agent
from pydantic_ai.models.openai import OpenAIModel
from pydantic_ai.providers.openai import OpenAIProvider
import os


def get_env_var(key: str) -> str:
    value: str | None = os.getenv(key)
    if not value:
        raise ValueError(f"{key} environment variable is not set.")
    return value


openwebui_uri: str = get_env_var("OPENWEBUI_URI")
openwebui_key: str = get_env_var("OPENWEBUI_API_KEY")
openwebui_client = AsyncOpenAI(
    base_url=f"{openwebui_uri}/api",
    api_key=openwebui_key,
)
openwebui_model: OpenAIModel = OpenAIModel(
    model_name="llama3.3:latest",
    provider=OpenAIProvider(openai_client=openwebui_client),
)

agent = Agent(model=openwebui_model)
result = await agent.run("ping")
print(result)

AgentRunResult(output='pong!')


In [ ]:
from ems_prepared.agents.models import openwebui_model, openwebui_uri, openwebui_key
from pydantic_ai import Agent

agent = Agent(model=openwebui_model)

result = await agent.run("ping")
print(result)

AgentRunResult(output='pong')


## Structured output

In [5]:
from pydantic import BaseModel
from pydantic_ai import Agent


# Define a simple Pydantic model
class Person(BaseModel):
    name: str
    age: int
    occupation: str
    location: str


# Create agent with structured output
structured_agent = Agent(model=openwebui_model, output_type=Person)

# Test the agent with a request for structured data
result = await structured_agent.run(
    "Create a person profile for a 28-year-old software engineer named Alice who lives in San Francisco"
)

print(f"Result type: {type(result)}")
print(f"Person data: {result}")
print(f"Name: {result.output.name}")
print(f"Age: {result.output.age}")
print(f"Occupation: {result.output.occupation}")
print(f"Location: {result.output.location}")

Result type: <class 'pydantic_ai.agent.AgentRunResult'>
Person data: AgentRunResult(output=Person(name='Alice', age=28, occupation='software engineer', location='San Francisco'))
Name: Alice
Age: 28
Occupation: software engineer
Location: San Francisco


## Comparisons

### Regular request

In [6]:
import requests

# Set up the request to OpenWebUI
headers = {
    "Authorization": f"Bearer {openwebui_key}",
    "Content-Type": "application/json",
}

payload = {
    "model": "llama3.3:latest",
    "messages": [{"role": "user", "content": "Hello, how are you?"}],
    "stream": False,
}

# Make the request
response = requests.post(
    f"{openwebui_uri}/api/chat/completions", headers=headers, json=payload
)

print(f"Status Code: {response.status_code}")
if response.status_code == 200:
    result = response.json()
    print(f"Response: {result['choices'][0]['message']['content']}")
else:
    print(f"Error: {response.text}")

Status Code: 200
Response: I'm doing well, thanks for asking! I'm a large language model, so I don't have feelings or emotions like humans do, but I'm always happy to chat and help with any questions or topics you'd like to discuss. How about you? How's your day going so far?


### commandline via curl

In [7]:
import subprocess
import json

result = subprocess.run(
    [
        "curl",
        f"{openwebui_uri}/ollama/api/generate",
        "-H",
        f"Authorization: Bearer {openwebui_key}",
        "-H",
        "Content-Type: application/json",
        '-d {"model": "llama3.2", "prompt": "What color is the sky at different times of the day? Respond using JSON", "format": "json", "stream": false }',
    ],
    capture_output=True,
)

if result.stdout:
    # Decode bytes to string and parse JSON
    response_text = result.stdout.decode("utf-8")
    response_json = json.loads(response_text)

    # print("Full Response:")
    # print(json.dumps(response_json, indent=2))

    print("\nExtracted Response Content:")
    if "response" in response_json:
        # Parse the nested JSON in the response field
        nested_response = json.loads(response_json["response"])
        print(json.dumps(nested_response, indent=2))

    print(f"\nModel: {response_json.get('model', 'N/A')}")
    print(f"Done: {response_json.get('done', 'N/A')}")
    print(f"Total Duration: {response_json.get('total_duration', 'N/A')} ns")
else:
    print("No output received")
    if result.stderr:
        print(f"Error: {result.stderr.decode('utf-8')}")


Extracted Response Content:
{
  "day": "Sky Color",
  "time": "AM/PM",
  "color": "Color"
}

Model: llama3.2
Done: True
Total Duration: 137867252 ns
